# MOML Project - Kaggle Direct Run Notebook

This notebook is ready to run in Kaggle as-is.

It will:
1. Find your uploaded project folder inside `/kaggle/input`.
2. Copy it to writable space: `/kaggle/working/MOML Project`.
3. Install dependencies from `requirements.txt`.
4. Run optimization + analysis + solution selection.
5. Zip outputs for download.

Before running, in Kaggle Notebook settings:
- Internet: ON
- Accelerator: GPU (optional, faster) or None (CPU)

In [ ]:
from pathlib import Path

print('Listing /kaggle/input (first few levels):')
for p in sorted(Path('/kaggle/input').glob('*')):
    print('-', p)

In [ ]:
from pathlib import Path

def find_project_root(base_dir='/kaggle/input'):
    base = Path(base_dir)
    candidates = []

    for req in base.rglob('requirements.txt'):
        root = req.parent
        if (root / 'scripts' / 'run_pipeline.sh').exists() and (root / 'configs' / 'default.yaml').exists():
            candidates.append(root)

    if not candidates:
        raise FileNotFoundError('Could not find project root under /kaggle/input. Make sure dataset contains the full repo.')

    candidates = sorted(candidates, key=lambda x: (len(str(x.parts)), str(x)))
    return candidates[0], candidates

PROJECT_SRC, ALL_CANDIDATES = find_project_root('/kaggle/input')
print('Detected project root:', PROJECT_SRC)
if len(ALL_CANDIDATES) > 1:
    print('Other candidates found:')
    for c in ALL_CANDIDATES[1:]:
        print('-', c)

In [ ]:
import os
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working/MOML Project')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

shutil.copytree(PROJECT_SRC, WORK_DIR)
os.chdir(WORK_DIR)

print('Working directory:', Path.cwd())
print('Top-level files:', sorted([p.name for p in Path.cwd().iterdir()])[:20])

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Dependencies installed successfully.')

In [ ]:
import os
import torch

os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"

print('PYTHONPATH set to include project root.')
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())

In [ ]:
from pathlib import Path
import yaml

cfg_path = Path('configs/default.yaml')
with cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Using full default config:', cfg_path)
print('Optimization trials:', cfg['optimization']['n_trials'])
print('Epoch range:', cfg['search_space']['epochs'])
print('Output directory:', cfg.get('output_dir', 'outputs'))

In [ ]:
import subprocess

subprocess.run(['bash', 'scripts/run_pipeline.sh', 'configs/default.yaml', 'outputs', '4'], check=True)

print('Full pipeline finished successfully.')

In [ ]:
import shutil
from pathlib import Path

archive_path = shutil.make_archive('/kaggle/working/moml_outputs', 'zip', root_dir='outputs')
print('Created archive:', archive_path)

print('Output files:')
for p in sorted(Path('outputs').rglob('*')):
    if p.is_file():
        print('-', p)